In [ ]:
# ── Source-of-truth note ──────────────────────────────────────────────────
#
# Earlier versions of this notebook held a string copy of src/ranker.py /
# src/report_generator.py in this cell and wrote it back to disk on every run.
# That pattern was a footgun: every "Run All" silently reverted the source file
# to whatever snapshot was frozen here, so improvements made directly in src/
# would disappear (e.g. Finding.to_dict, profile re-ranking, the numeric
# grounding validator, prompt caching).
#
# The notebook now treats src/ as the single source of truth. The cells below
# IMPORT from src/, they do not generate or overwrite it.
print("Using src/ranker.py and src/report_generator.py as the source of truth.")


In [ ]:
import sys, os, json, importlib, logging
from pathlib import Path
from dotenv import load_dotenv

# Show INFO logs from the ranker and report generator
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

sys.path.insert(0, str(Path("..").resolve() / "src"))

import report_generator
importlib.reload(report_generator)
from report_generator import ReportGenerator, PROFILE_GROWTH, PROFILE_SCALE

def find_project_root(start):
    for p in [start, *start.parents]:
        if (p / "data" / "metrics_159d.csv").exists():
            return p
    raise FileNotFoundError("Cannot find data/metrics_159d.csv")

ROOT = find_project_root(Path.cwd())
load_dotenv(ROOT / ".env")

print(f"Root: {ROOT}")
print("Imports successful")


In [ ]:
with open(ROOT / "outputs" / "ranked_findings.json", encoding="utf-8") as f:
    ranked = json.load(f)

print("Dates available:")
print("  Daily  :", list(ranked["daily"].keys()))
print("  Weekly :", list(ranked["weekly"].keys()))

In [ ]:
gen = ReportGenerator()
OUTPUT = ROOT / "outputs" / "reports"
OUTPUT.mkdir(exist_ok=True)

daily_dates = list(ranked["daily"].keys())
# Pair: same date with growth profile AND scale profile (shows personalisation),
# plus two more dates with growth profile to fill 4 total.
plan = [
    (daily_dates[0], PROFILE_GROWTH),
    (daily_dates[0], PROFILE_SCALE),    # same date, different profile = visible diff
    (daily_dates[1], PROFILE_GROWTH),
    (daily_dates[2], PROFILE_SCALE),
]

steady_by_date = ranked.get("daily_steady", {})
report_paths = []

for i, (date, profile) in enumerate(plan):
    print(f"\nGenerating daily digest {i+1}/{len(plan)} — {date} — {profile.name}...")
    findings = ranked["daily"][date]
    steady   = steady_by_date.get(date, [])
    report   = gen.generate_daily(findings, profile, date, steady=steady)

    slug = profile.name.split()[0].lower()
    path = OUTPUT / f"daily_{date}_{slug}.md"
    gen.save_report(report, path)
    report_paths.append(path)

    preview = "\n".join(report.split("\n")[:8])
    print(preview)
    print("...")

print(f"\n{len(plan)} daily digests saved to {OUTPUT}")


In [ ]:
weekly_dates = list(ranked["weekly"].keys())
profiles_w   = [PROFILE_GROWTH, PROFILE_SCALE]
steady_weekly = ranked.get("weekly_steady", {})

# Build simple daily summaries for weekly context
daily_summaries = {
    date: ranked["daily"][date][0]["fact_sentence"]
    for date in ranked["daily"].keys()
}

import pandas as pd

for week_end, profile in zip(weekly_dates, profiles_w):
    print(f"\nGenerating weekly report — week ending {week_end} — {profile.name}...")
    findings = ranked["weekly"][week_end]
    steady   = steady_weekly.get(week_end, [])

    week_start = str((pd.Timestamp(week_end) - pd.Timedelta(days=6)).date())
    summaries  = [v for k, v in daily_summaries.items() if week_start <= k <= week_end]

    report = gen.generate_weekly(findings, profile, week_end, summaries, steady=steady)

    slug = profile.name.split()[0].lower()
    path = OUTPUT / f"weekly_{week_end}_{slug}.md"
    gen.save_report(report, path)

    preview = "\n".join(report.split("\n")[:10])
    print(preview)
    print("...")

print(f"\nAll weekly reports saved.")


In [ ]:
# Read and display the first daily digest in full
first_report = report_paths[0]
content = first_report.read_text(encoding="utf-8")
print(content)


In [ ]:
import os
from pathlib import Path

def find_project_root(start):
    for p in [start, *start.parents]:
        if (p / "data" / "metrics_159d.csv").exists():
            return p

ROOT    = find_project_root(Path.cwd())
reports = sorted((ROOT / "outputs" / "reports").glob("*.md"))

print(f"Total reports generated: {len(reports)}")
print()
for r in reports:
    size = r.stat().st_size
    print(f"  {r.name:45s}  {size:,} bytes")